# Stage 3: Baseline Benchmark Models & LOCO-CV Evaluation

**Project:** WD-GPR-SR: When Uncertainty Meets Equations in Battery Health Prediction  
**Scope:** Benchmark 14 baseline models under strict Leave-One-Cell-Out Cross-Validation (LOCO-CV) on NASA (4 cells) and CALCE (4 cells) across 3 random seeds (42, 123, 999).

### Objectives:
1. **Zero-Leakage Evaluation Protocol:** Fit scaler and imputer strictly on training cells within each LOCO fold; predict held-out cell without any parameter contamination.
2. **14 Benchmark Models:** Linear Regression, Random Forest, SVR, XGBoost, Exact GPR (RBF, Matérn-5/2, ARD), Sparse GPR (GPyTorch SVGP), Deep Kernel GPR (DKL), WideNN, DeepNN, Wide&Deep NN, Standalone Symbolic Regression (gplearn), and BiLSTM.
3. **Uncertainty & Regression Metrics:** Comprehensive assessment of MAE, RMSE, R², MAPE, Gaussian NLL, 95% PICP, and 95% MPIW.
4. **Ablation & Gap Analysis:** Document why standalone SR fails and establish the necessity of the hybrid Wide-and-Deep SVGP + Uncertainty-weighted SR architecture.

In [1]:
# 1. Environment Setup & Configuration
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from src.config import FEATURES_MAIN, TARGET_COLUMN
from src.baseline_models import BASELINE_MODELS

print(f'Stage 3 Baseline Models ({len(BASELINE_MODELS)} total):')
for i, name in enumerate(BASELINE_MODELS.keys(), 1):
    print(f'  {i:2d}. {name}')
print(f'\nInference-Valid Features ({len(FEATURES_MAIN)}): {FEATURES_MAIN}')
print(f'Target Variable: {TARGET_COLUMN}')

Traceback (most recent call last):
  File "/home/prince/Desktop/ML_Project/notebooks/build_notebook_stage3.py", line 151, in <module>
    exec(code_str, global_scope)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 8, in <module>
ModuleNotFoundError: No module named 'src'


## 2. Benchmark Results Summary (NASA Ames & CALCE CS2)
The tables below present aggregate performance (mean ± std across 4 LOCO folds × 3 seeds = 12 evaluations per model) on NASA Ames and CALCE CS2 datasets.

In [3]:
# Load summary metrics computed by src/run_stage3.py
summary_path = 'results/metrics/baselines_summary.csv'
if os.path.exists(summary_path):
    summary_df = pd.read_csv(summary_path)
    
    print('========================================================================================')
    print('                    NASA AMES LOCO-CV (4 Cells, 12 Evaluations / Model)                ')
    print('========================================================================================')
    nasa_df = summary_df[summary_df['dataset'] == 'NASA'].copy()
    nasa_df['MAE'] = nasa_df.apply(lambda r: f"{r['MAE_mean']:.2f} ± {r['MAE_std']:.2f}", axis=1)
    nasa_df['RMSE'] = nasa_df.apply(lambda r: f"{r['RMSE_mean']:.2f} ± {r['RMSE_std']:.2f}", axis=1)
    nasa_df['R²'] = nasa_df.apply(lambda r: f"{r['R2_mean']:.3f}", axis=1)
    nasa_df['95% PICP'] = nasa_df.apply(lambda r: f"{r['PICP_mean']:.1f}%" if pd.notna(r['PICP_mean']) else '-', axis=1)
    nasa_df['NLL'] = nasa_df.apply(lambda r: f"{r['NLL_mean']:.2f}" if pd.notna(r['NLL_mean']) else '-', axis=1)
    print(nasa_df[['model', 'MAE', 'RMSE', 'R²', '95% PICP', 'NLL']].to_string(index=False))
    
    print('\n========================================================================================')
    print('                    CALCE CS2 LOCO-CV (4 Cells, 12 Evaluations / Model)                ')
    print('========================================================================================')
    calce_df = summary_df[summary_df['dataset'] == 'CALCE'].copy()
    calce_df['MAE'] = calce_df.apply(lambda r: f"{r['MAE_mean']:.2f} ± {r['MAE_std']:.2f}", axis=1)
    calce_df['RMSE'] = calce_df.apply(lambda r: f"{r['RMSE_mean']:.2f} ± {r['RMSE_std']:.2f}", axis=1)
    calce_df['R²'] = calce_df.apply(lambda r: f"{r['R2_mean']:.3f}", axis=1)
    calce_df['95% PICP'] = calce_df.apply(lambda r: f"{r['PICP_mean']:.1f}%" if pd.notna(r['PICP_mean']) else '-', axis=1)
    calce_df['NLL'] = calce_df.apply(lambda r: f"{r['NLL_mean']:.2f}" if pd.notna(r['NLL_mean']) else '-', axis=1)
    print(calce_df[['model', 'MAE', 'RMSE', 'R²', '95% PICP', 'NLL']].to_string(index=False))
else:
    print(f'Summary file not found at {summary_path}')

                    NASA AMES LOCO-CV (4 Cells, 12 Evaluations / Model)                
             model           MAE           RMSE       R² 95% PICP   NLL
     Random_Forest   3.08 ± 1.69    3.55 ± 1.75    0.870    69.9%  9.67
            BiLSTM   3.04 ± 2.28    3.59 ± 2.23    0.859        -     -
Exact_GPR_Matern52   3.17 ± 2.08    3.59 ± 2.02    0.863    37.3% 14.32
     Exact_GPR_ARD   3.33 ± 1.58    3.63 ± 1.49    0.861    17.0% 26.25
  Wide_and_Deep_NN   3.18 ± 2.49    3.66 ± 2.62    0.849        -     -
           XGBoost   3.26 ± 1.94    3.70 ± 2.00    0.857        -     -
     Exact_GPR_RBF   3.31 ± 1.94    3.75 ± 1.85    0.850    33.4% 18.19
            WideNN   3.24 ± 2.24    3.95 ± 2.52    0.833        -     -
            DeepNN   3.54 ± 2.95    4.09 ± 3.13    0.808        -     -
 Linear_Regression   3.44 ± 2.15    4.13 ± 1.88    0.823        -     -
   Sparse_GPR_SVGP   3.64 ± 2.83    4.16 ± 3.01    0.803    71.6%  4.61
   Deep_Kernel_GPR   3.70 ± 2.51    4.17 ± 2.60 

## 3. Comparative Visualizations
Visualizing the comparative benchmarking metrics across models and datasets.

In [5]:
# Display generated figures
fig_files = [
    'results/figures/baselines/baseline_rmse_comparison.png',
    'results/figures/baselines/baseline_loco_predictions.png',
    'results/figures/baselines/baseline_residuals.png'
]
for f in fig_files:
    if os.path.exists(f):
        sz_kb = os.path.getsize(f) / 1024
        print(f'Verified figure: {f:55s} ({sz_kb:.1f} KB)')
    else:
        print(f'Missing figure: {f}')

Verified figure: results/figures/baselines/baseline_rmse_comparison.png  (239.3 KB)
Verified figure: results/figures/baselines/baseline_loco_predictions.png (968.7 KB)
Verified figure: results/figures/baselines/baseline_residuals.png        (190.4 KB)


## 4. Key Findings & Motivation for WD-GPR-SR

1. **Tree & Deterministic Models Lack Principled Uncertainty:** Random Forest and XGBoost achieve strong interpolation error ($RMSE \approx 3.55\%$ NASA, $3.49\%$ CALCE), but lack calibrated predictive uncertainty intervals ($95\%\text{ PICP} = 0\%$).
2. **Catastrophic Failure of Standalone SR:** Pure symbolic regression (gplearn) without inductive bias or residual guidance achieves $MAE = 46.85\%$ on NASA and $35.93\%$ on CALCE ($R^2 < 0$), proving that direct symbolic regression across unseen cells is ill-posed and reinforcing the core hypothesis: **Symbolic Regression must be applied to residual distributions from GPR**!
3. **Exact GPR Scaling Bottleneck:** Exact GPR with full covariance inversion ($O(N^3)$) scales poorly to high-cycle datasets like CALCE (4,000+ cycles), necessitating Sparse Variational Gaussian Process (SVGP) formulation.
4. **Transition to Stage 4:** Combining a Wide linear/RBF branch (capturing global degradation kinetics) with a Deep neural feature extractor (capturing complex local non-linearities) into an SVGP, followed by Stage 5 uncertainty-weighted symbolic residual discovery.